# Day 27 — vLLM Internals

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pradeepvaka/llm-inference-90day/blob/master/notebooks/day27-vllm-internals.ipynb)

**Big idea:** vLLM is an operating system for inference — the scheduler is the process scheduler, the block manager is the virtual memory manager, the model runner executes one timeslice per tick.
CPU runs cells 1–9 (~40 min). Cell 10 is the optional T4 live burst test with scheduler debug prints.


In [ ]:
# Colab: install deps (only cell that installs)
%pip install --quiet torch
print('deps ok')
# Expected: deps ok


## 1. Setup
Pure PyTorch simulation of the scheduler + block manager. No model downloads, no vLLM install needed for cells 1–9.


In [ ]:
import math
import torch
from dataclasses import dataclass, field

torch.manual_seed(27)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device: {device}")
# Expected: device: cpu  (or cuda on a T4 runtime)


## 2. KV-cache capacity math — where does preemption kick in?
The Day-27 worked example for Llama-3.1-8B (32 layers, 8 KV heads, head dim 128, fp16) on a 16 GB T4 with 90% KV reserve, block size 16.


In [ ]:
layers, kv_heads, head_dim = 32, 8, 128
bytes_per_elem, block_size = 2, 16
kv_bytes_per_token = 2 * layers * kv_heads * head_dim * bytes_per_elem
print(f"KV bytes/token: {kv_bytes_per_token} = {kv_bytes_per_token/1024:.1f} KiB")

block_bytes = kv_bytes_per_token * block_size
print(f"bytes/physical block ({block_size} tokens): {block_bytes/2**20:.1f} MiB")

gpu_kv_pool = 0.90 * 16 * 2**30
num_blocks = int(gpu_kv_pool // block_bytes)
capacity_tokens = num_blocks * block_size
print(f"physical blocks: {num_blocks}  ->  capacity: {capacity_tokens:,} tokens")

# How many concurrent sequences at avg context 1800 tokens before preemption?
demand_per_seq = 1800
max_seqs = capacity_tokens // demand_per_seq
print(f"preemption threshold: ~{max_seqs} concurrent seqs at 1800-token contexts")
# Expected:
# KV bytes/token: 131072 = 128.0 KiB
# bytes/physical block (16 tokens): 2.0 MiB
# physical blocks: 7372  ->  capacity: 117,952 tokens
# preemption threshold: ~65 concurrent seqs at 1800-token contexts


## 3. Token-budget simulator — chunked prefill under a 512-token quantum
Mini `_schedule()`: FCFS admission from the waiting queue, each waiting prompt contributes a prefill chunk (`max_prefill_chunk=256`), each running decode contributes exactly 1 token. Watch a 600-token prompt get chunked so decodes never stall.


In [ ]:
@dataclass
class SimSeq:
    sid: int
    prompt_left: int
    generated: int = 0
    preempted: bool = False

MAX_BATCHED, PREFILL_CHUNK = 512, 256
waiting = [SimSeq(sid=i, prompt_left=600) for i in range(3)]
running = [SimSeq(sid=10 + i, prompt_left=0) for i in range(20)]

for tick in range(4):
    batched, plan = 0, []
    # running decodes: 1 token each
    for s in running:
        batched += 1; s.generated += 1
    # FCFS waiting: prefill chunks
    for s in waiting:
        if batched >= MAX_BATCHED: break
        chunk = min(s.prompt_left, PREFILL_CHUNK, MAX_BATCHED - batched)
        if chunk <= 0: break
        batched += chunk; s.prompt_left -= chunk; plan.append((s.sid, chunk))
        if s.prompt_left == 0:   # prefill done -> becomes a running decode
            running.append(s)
    waiting = [s for s in waiting if s.prompt_left > 0]
    print(f"tick {tick}: Running: {len(running)} seqs, {batched} tokens; Waiting: {len(waiting)}; Swapped: 0")
# Expected (tick 0: 600-token prompts chunked, decodes unaffected):
# tick 0: Running: 20 seqs, 512 tokens; Waiting: 3; Swapped: 0
# ...the 600-token prompt enters running over ~3 ticks, decodes never stall


## 4. Block manager — logical-to-physical page tables
Toy `BlockManagerV2`: allocate logical blocks -> physical blocks, append slots as tokens generate, free on finish. Plus prefix-cache sharing: two sequences with the same prompt prefix share one physical block (refcount > 1), like copy-on-write fork.


In [ ]:
class BlockManager:
    def __init__(self, n_blocks, block_size=16):
        self.block_size, self.n_blocks = block_size, n_blocks
        self.free = list(range(n_blocks))
        self.table = {}      # seq_id -> [physical block ids]
        self.refcount = {}   # physical block -> refcount

    def allocate(self, sid, n_tokens):
        need = math.ceil(n_tokens / self.block_size)
        if len(self.free) < need: return False
        self.table[sid] = [self.free.pop() for _ in range(need)]
        for b in self.table[sid]: self.refcount[b] = 1
        return True

    def append_slots(self, sid, n_tokens=1):
        # one new block per block_size tokens
        cur = self.table[sid]
        if (n_tokens % self.block_size) == 0 or not cur:  # simplified boundary
            pass
        return len(cur)

    def free_seq(self, sid):
        for b in self.table.pop(sid, []):
            self.refcount[b] -= 1
            if self.refcount[b] == 0: self.free.append(b)

bm = BlockManager(n_blocks=100)
ok_a = bm.allocate(sid='A', n_tokens=40)   # 3 blocks
ok_b = bm.allocate(sid='B', n_tokens=20)   # 2 blocks
print(f"allocated A: {ok_a}, blocks {bm.table['A']}")
print(f"allocated B: {ok_b}, blocks {bm.table['B']}")
print(f"free blocks left: {len(bm.free)}")

# prefix-cache sharing: B's first block IS A's first block (shared prefix)
shared = bm.table['A'][0]
bm.free.append(bm.table['B'][0]); bm.refcount.pop(bm.table['B'][0])
bm.table['B'][0] = shared; bm.refcount[shared] = 2
print(f"shared block {shared} refcount: {bm.refcount[shared]} (Day 25 prefix caching = this)")

bm.free_seq('A')
print(f"after freeing A: shared block {shared} refcount: {bm.refcount[shared]}, free: {len(bm.free)}")
# Expected:
# allocated A: True, blocks [0, 1, 2]  (order may vary)
# allocated B: True, blocks [3, 4]
# free blocks left: 95
# shared block 0 refcount: 2 (Day 25 prefix caching = this)
# after freeing A: shared block 0 refcount: 1, free: 96


## 5. Reading a schedule log — from numbers to narration
Given `Running / tokens / Waiting / Swapped` per tick plus the token budget, compute budget utilization and classify the engine state: healthy, memory-bound, or thrashing (swapping pages instead of generating tokens).


In [ ]:
def narrate(tick, running, tokens, waiting, swapped, budget=2048):
    util = tokens / budget
    if swapped > 0 and util < 0.5:
        state = "THRASHING: pages moving, not tokens"
    elif waiting > 0 and util >= 0.9:
        state = "healthy but saturated: budget-bound, waiting FCFS"
    elif waiting > 0 and util < 0.9:
        state = "memory-bound: blocks short, waiting starved"
    else:
        state = "healthy: budget has headroom"
    print(f"tick {tick}: Running {running} seqs / {tokens} tokens ({util:.0%} of budget), "
          f"Waiting {waiting}, Swapped {swapped} -> {state}")

log = [(0,12,1536,3,1),(1,12,1536,4,2),(2,14,1900,2,0),(3,8,900,0,0)]
for t in log: narrate(*t)
# Expected:
# tick 0: ... (75% of budget), Waiting 3, Swapped 1 -> memory-bound: blocks short, waiting starved
# tick 1: ... THRASHING ...
# tick 2: ... healthy but saturated ...
# tick 3: ... healthy: budget has headroom


## 6. Shallow-clone vLLM and touch the real scheduler
Colab shell cells: clone shallow, then grep for the exact budget check and preemption trigger you annotated in the packet. (Read-only; no install.)


In [ ]:
!git clone --depth 1 https://github.com/vllm-project/vllm vllm-src 2>&1 | tail -1
!grep -n "max_num_batched_tokens" vllm-src/vllm/core/scheduler.py | head -5
print("---")
!grep -n "def _preempt" vllm-src/vllm/core/scheduler.py
print("---")
!grep -n "def allocate" vllm-src/vllm/core/block_manager_v2.py
# Expected: clone line, 3-5 hits for the budget check, one _preempt def, one allocate def


## 7. The engine loop — five beats per tick
`LLMEngine.step()`: schedule -> prepare inputs -> model forward -> sample -> postprocess. Trace the call chain from `add_request` to output tokens in your notes; this cell just fixes the chain in code.


In [ ]:
chain = [
    ("vllm/engine/llm_engine.py", "add_request / step"),
    ("vllm/core/scheduler.py", "_schedule: budget + FCFS + preempt"),
    ("vllm/core/block_manager_v2.py", "allocate / append_slots"),
    ("vllm/worker/model_runner.py", "execute_model: one forward pass"),
    ("vllm/engine/llm_engine.py", "sample + postprocess"),
]
for f, fn in chain:
    print(f"{f:45s} {fn}")
print("\nbeats per tick: schedule, prepare, forward, sample, postprocess")
# Expected: the 5-line chain above


## 8. vllm-arch.md starter + the 10x batch-workload exercise
Copy the Mermaid block into `vllm-arch.md` and annotate one line per box. Then write your 3 changes for a 10x-throughput batch workload (no latency SLO) — the packet suggests: bigger token/seq budgets, disable chunked prefill, disable preemption in favor of queueing.


In [ ]:
mermaid = '''
```mermaid
graph LR
    A[client: OpenAI API] --> B[engine: llm_engine.py step]
    B --> C[scheduler: _schedule]
    C --> D[block manager v2: allocate]
    D --> E[model runner: execute_model]
    E --> F[sampler: logits -> tokens]
    F --> B
```
'''
print(mermaid)
print("Exercise: which single config change buys the most batch throughput, and why?")
# Expected: the mermaid graph prints; your answer: max_num_batched_tokens up — fills the GPU


## 9. Checkpoint — answer from memory, then check the packet
1. Which component owns the logical→physical block mapping? 2. Where is the per-iteration token budget enforced? 3. A sequence marked PREEMPTED goes where? 4. Why does a preempted decode cost a full re-prefill? 5. 12 seqs / 1536 tokens on a 2048 budget — utilization and implication?


In [ ]:
answers = {
  1: "block manager v2 (scheduler asks it to allocate)",
  2: "scheduler._schedule() via max_num_batched_tokens",
  3: "back to the waiting queue; blocks freed or swapped to CPU",
  4: "KV pages evicted -> KV must be recomputed from prompt + generated tokens",
  5: "75%; check Waiting: high waiting + high util = budget-bound, high waiting + low util = block-bound",
}
print("answers hidden in packet section 6; verify yours before peeking:")
for k in sorted(answers): print(f"  {k}. {answers[k]}")
# Expected: the 5 answers above


## 10. (T4 optional) Live burst test with scheduler debug prints
Switch to a T4 GPU runtime, install vLLM, patch `_schedule()` with the `Running/Waiting/Swapped` print, and run a Day-20-style burst. Heavy cell — run only if you have a T4 and ~20 min.


In [ ]:
# OPTIONAL T4 cell — skip on CPU
# %pip install --quiet vllm
# from vllm import LLM, SamplingParams
# llm = LLM(model="facebook/opt-125m", max_num_batched_tokens=512, max_num_seqs=16)
# prompts = [f"Summarize: {'the quick brown fox jumps over the lazy dog. '*20}"] * 30
# outs = llm.generate(prompts, SamplingParams(max_tokens=40, temperature=0))
# print(f"completed {len(outs)} requests")
# Expected on T4: 30 completions; with the debug print patched into _schedule(),
# watch Swapped rise when blocks run out
